# TP Séance 1 — Comparaison de modèles open-source (exécution locale sur Colab)

**Objectif :** sur un même jeu de prompts, faire répondre plusieurs LLM ouverts et comparer leurs réponses (exactitude, clarté, créativité, rapidité).

**Outils :** Google Colab (GPU gratuit), `transformers`. Aucun token ni crédit n'est nécessaire, les modèles sont téléchargés et exécutés directement sur la machine Colab.

**Avant de commencer :** `Exécution > Modifier le type d'exécution > GPU (T4)`.

In [ ]:
!pip install -q transformers accelerate

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device :", DEVICE)
if DEVICE == "cpu":
    print(
        "Activez le GPU : Exécution > Modifier le type d'exécution > GPU (sinon le TP sera très lent)."
    )

## 1. Configuration

Modèles ouverts et non « gated » (aucune acceptation de licence requise).

In [ ]:
MODELS = {
    "Qwen 2.5 1.5B": "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen 2.5 3B": "Qwen/Qwen2.5-3B-Instruct",
    "Phi 3.5 mini": "microsoft/Phi-3.5-mini-instruct",
}

PROMPTS = {
    "Factuel": "Quelle est la capitale de l'Australie et depuis quand le est-elle ?",
    "Créatif": "Écris un slogan pour une application mobile qui aide les étudiants à réviser leurs examens.",
    "Raisonnement": "Un train part de Grenoble à 8h00 à 90 km/h. Un second part de Lyon (110 km plus loin) "
    "à 8h15 à 110 km/h en direction de Grenoble. À quelle heure se croisent-ils ?",
}

MAX_NEW_TOKENS = 300

## 2. Fonctions d'inférence

In [ ]:
import time
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM


def load_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        device_map="auto",
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer, model


def ask_model(tokenizer, model, prompt, max_new_tokens=MAX_NEW_TOKENS):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)

    start = time.time()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    elapsed = round(time.time() - start, 2)

    new_tokens = output_ids[0][inputs["input_ids"].shape[1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip(), elapsed

## 3. Comparaison des modèles

Chaque modèle est chargé, interrogé sur les 3 prompts, puis libéré de la mémoire GPU avant de passer au suivant.

In [ ]:
results = []

for model_name, model_id in MODELS.items():
    print(f"Chargement de {model_name} ({model_id})...")
    try:
        tokenizer, model = load_model(model_id)
    except Exception as e:
        results.append(
            {
                "Modèle": model_name,
                "Prompt": "-",
                "Temps (s)": None,
                "Réponse": f"[Erreur de chargement : {type(e).__name__}: {str(e)[:200]}]",
            }
        )
        continue

    for prompt_name, prompt in PROMPTS.items():
        answer, elapsed = ask_model(tokenizer, model, prompt)
        results.append(
            {
                "Modèle": model_name,
                "Prompt": prompt_name,
                "Temps (s)": elapsed,
                "Réponse": answer,
            }
        )

    del model, tokenizer
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print("Terminé.")

In [ ]:
df = pd.DataFrame(results)
df.to_csv("resultats_modeles.csv", index=False)
df

## 4. Lecture comparative

Lisez les réponses prompt par prompt pour comparer les modèles côte à côte.

In [ ]:
for prompt_name in PROMPTS:
    print(f"===== {prompt_name} =====")
    for _, row in df[df["Prompt"] == prompt_name].iterrows():
        print(f"\n--- {row['Modèle']} ({row['Temps (s)']} s) ---")
        print(row["Réponse"])
    print()

## 5. Grille d'observation

Notez chaque modèle de 1 à 5 sur chaque critère (à compléter après lecture des réponses).

| Modèle | Prompt | Exactitude | Clarté | Créativité | Rapidité (temps) | Commentaire |
|---|---|---|---|---|---|---|
| Qwen 2.5 1.5B | Factuel | | | | | |
| Qwen 2.5 3B | Factuel | | | | | |
| Phi 3.5 mini | Factuel | | | | | |


### 1.4 — Modèles propriétaires (GPT, Claude, Gemini)

Ces modèles ne sont pas gratuits via API. Testez-les **manuellement**, avec les mêmes 3 prompts, via leurs interfaces gratuites :

| Modèle | Interface gratuite |
|---|---|
| GPT | [chatgpt.com](https://chatgpt.com) |
| Claude | [claude.ai](https://claude.ai) |
| Gemini | [gemini.google.com](https://gemini.google.com) |

Consignez vos observations dans le tableau ci-dessous (exécutez la cellule puis éditez les valeurs).

In [ ]:
# 1.5 — Grille de comparaison à compléter manuellement (modèles propriétaires)
grille_manuelle = pd.DataFrame(
    {
        "Critère": ["Exactitude", "Clarté", "Créativité", "Rapidité perçue"],
        "GPT": ["", "", "", ""],
        "Claude": ["", "", "", ""],
        "Gemini": ["", "", "", ""],
    }
)
# Exemple pour éditer une cellule : grille_manuelle.loc[0, "GPT"] = "5/5 - réponse correcte et sourcée"
grille_manuelle

### 💬 Questions de synthèse TP1
- Quel modèle a été le plus rapide ? Le plus précis sur le prompt de raisonnement ?
- Avez-vous observé de grandes différences de style entre modèles ouverts et propriétaires ?
- Sur quel(s) critère(s) choisiriez-vous un modèle plutôt qu'un autre pour un usage professionnel ?

---
## TP2 — Analyse des hallucinations (~30 min)

**Objectif :** apprendre à **provoquer**, **repérer** et **contrer** une hallucination — une réponse formulée avec assurance mais factuellement fausse ou inventée.

In [ ]:
# 2.1 — Prompts conçus pour provoquer une hallucination
prompts_pieges = [
    "Cite l'article exact et le numéro de page où Victor Hugo mentionne l'intelligence artificielle dans Les Misérables.",
    "Qui a remporté la Coupe du Monde de football en 2030 ?",
    "Donne-moi le titre et la date de publication du rapport officiel de l'ONU sur la population de la Lune.",
    "Je veux laver ma voiture à la station service qui se trouve à 50 m. Dois-je y aller à pieds ou en vélo ?",
    "Explique en détail comment Napoléon Bonaparte a utilisé les réseaux sociaux et la messagerie instantanée Telegram pour organiser sa campagne d'Égypte en 1798.",
    "Rédige un court paragraphe sur l'importance de boire de l'eau. Mais attention : tu n'as pas le droit d'utiliser une seule fois la lettre 'e' (ni majuscule, ni minuscule), ni d'utiliser le mot 'eau' ou 'boire.",
]

modele_test = (
    "Qwen 2.5 3B"  # vous pouvez changer de modèle (Qwen 2.5 1.5B, Phi 3.5 mini)
)

_cache = {}


def ask_local(prompt, model_name):
    if model_name not in _cache:
        _cache.clear()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        _cache[model_name] = load_model(MODELS[model_name])
    tokenizer, model = _cache[model_name]
    return ask_model(tokenizer, model, prompt)


reponses_hallucinations = []
for p in prompts_pieges:
    answer, elapsed = ask_local(p, modele_test)
    reponses_hallucinations.append(
        {"Prompt piège": p, "Modèle": modele_test, "Réponse": answer}
    )
    print(f"— {p}\n→ {answer}\n")

df_hallucinations = pd.DataFrame(reponses_hallucinations)
df_hallucinations

### 2.2 — Repérer

Pour chaque réponse obtenue ci-dessus :
- Vérifiez chaque affirmation factuelle (recherche web, source fiable).
- Cherchez les signaux de « fausse confiance » (détails inventés très précis, absence de nuance).
- Comparez avec la réponse d'un second modèle sur le même prompt.

In [ ]:
# 2.3 — Journal d'analyse à compléter
journal_hallucinations = pd.DataFrame(
    {
        "Prompt piège": [p[:40] + "…" for p in prompts_pieges],
        "Hallucination détectée ? (Oui/Non)": ["", "", ""],
        "Élément(s) inventé(s)": ["", "", ""],
        "Comment la contrer (reformulation, RAG, contrainte...)": ["", "", ""],
    }
)
journal_hallucinations

### 2.4 — Contrer

Reformulez l'un des prompts pièges pour réduire le risque d'hallucination (ex. demander explicitement une source, autoriser le modèle à dire « je ne sais pas », limiter la portée de la question).

In [ ]:
prompt_original = prompts_pieges[0]
prompt_reformule = (
    "Si tu ne connais pas la réponse avec certitude, réponds explicitement "
    "\"Je ne dispose pas d'information fiable à ce sujet\" plutôt que d'inventer une source. "
    + prompt_original
)

reponse_originale, _ = ask_local(prompt_original, modele_test)
reponse_reformulee, _ = ask_local(prompt_reformule, modele_test)

print("AVANT :\n", reponse_originale)
print("\nAPRÈS :\n", reponse_reformulee)

### 💬 Questions de synthèse TP2
- La reformulation a-t-elle réduit l'hallucination ? Pourquoi (ou pourquoi pas) selon vous ?
- Quelle autre stratégie du cours (RAG, comparaison multi-modèles…) pourrait aider ici ?

---
## TP3 — Évaluation de prompts (~30 min)

**Objectif :** réécrire un prompt faible et mesurer l'amélioration obtenue, à l'aide d'une grille d'évaluation.

**Grille d'évaluation (1 = absent, 5 = excellent) :** Clarté · Contexte fourni · Contraintes de format · Exemples · Rôle assigné

In [ ]:
prompt_faible = "Parle-moi de l'IA"

prompt_ameliore = (
    "Tu es formateur en école de commerce. Explique l'IA générative en 3 points "
    "à des étudiants de Master non-techniques, avec un exemple concret par point."
)

modele_eval = (
    "Qwen 2.5 3B"  # vous pouvez changer de modèle (Qwen 2.5 1.5B, Phi 3.5 mini)
)

reponse_faible, temps_faible = ask_local(prompt_faible, modele_eval)
reponse_amelioree, temps_amelioree = ask_local(prompt_ameliore, modele_eval)

print("=== Prompt faible ===")
print(reponse_faible)
print(f"\n(temps: {temps_faible}s)\n")
print("=== Prompt amélioré ===")
print(reponse_amelioree)
print(f"\n(temps: {temps_amelioree}s)")

In [ ]:
# 3.2 — Notez chaque prompt sur la grille d'évaluation (1 à 5)
grille_evaluation = pd.DataFrame(
    {
        "Critère": [
            "Clarté",
            "Contexte fourni",
            "Contraintes de format",
            "Exemples",
            "Rôle assigné",
        ],
        "Prompt faible": [None, None, None, None, None],
        "Prompt amélioré": [None, None, None, None, None],
    }
)
# Exemple : grille_evaluation.loc[0, "Prompt faible"] = 1
grille_evaluation

In [ ]:
# 3.3 — Score total (à exécuter une fois la grille remplie ci-dessus)
if grille_evaluation[["Prompt faible", "Prompt amélioré"]].isnull().values.any():
    print(
        "⚠️  Complétez d'abord la grille ci-dessus (remplacez les None par une note de 1 à 5)."
    )
else:
    totaux = grille_evaluation[["Prompt faible", "Prompt amélioré"]].sum()
    print("Score total sur 25 :")
    print(totaux)

### 3.4 — À votre tour

Choisissez un prompt faible de votre choix (lié à un domaine qui vous intéresse : RH, marketing, finance…), améliorez-le en appliquant la grille, et comparez les deux réponses.

In [ ]:
mon_prompt_faible = ""  # à compléter
mon_prompt_ameliore = ""  # à compléter

if mon_prompt_faible and mon_prompt_ameliore:
    r1, _ = ask_local(mon_prompt_faible, modele_eval)
    r2, _ = ask_local(mon_prompt_ameliore, modele_eval)
    print("=== Votre prompt faible ===\n", r1)
    print("\n=== Votre prompt amélioré ===\n", r2)
else:
    print(
        "Complétez mon_prompt_faible et mon_prompt_ameliore ci-dessus puis relancez la cellule."
    )

### 💬 Questions de synthèse TP3
- Quel critère de la grille a le plus fait progresser la qualité de la réponse ?
- Ce résultat serait-il différent avec un autre modèle ? Pourquoi ?

---
## Synthèse de la séance

- Un LLM traite des **tokens**, pas des mots.
- Les **embeddings** encodent le sens sous forme de vecteurs.
- La **fenêtre de contexte** limite ce que le modèle peut « voir ».
- L'**attention** pondère l'importance des mots du contexte.
- Aucun modèle n'est infaillible : **vérifier reste indispensable**.

### Pour la Séance 2
- Créez vos comptes Google Colab et Hugging Face si ce n'est pas déjà fait.
- Relisez vos notes sur le RAG et les embeddings.
- Apportez le résultat de ce TP1 (comparaison de modèles).

**Séance 2 : Construction d'une application LLM** — Prompt Engineering, RAG → TP : création d'un chatbot documentaire (LangChain + ChromaDB + Gradio).